# DYAD 채보 생성 (Mapperatorinator · osu!taiko)

음원 하나를 넣으면 [Mapperatorinator](https://github.com/OliBomby/Mapperatorinator)(V32) 로 **EASY / NORMAL / HARD** 세 개의 taiko 채보를 만들고,
DYAD 의 `songs-src/<id>/` 폴더 형식 그대로 묶어서 내려줍니다.

```
<id>/
  easy.osu  normal.osu  hard.osu   ← 세 채보
  song.wav                         ← 빌드 입력 (git 에는 넣지 않음)
  song.json                        ← { title, artist, audioOffset: 0 }
```

내려받은 폴더를 저장소의 `songs-src/` 에 넣고 `jacket.png|jpg|webp` 를 옆에 두면, `npm run songs:build` 를 돌리면 `public/songs/<id>/` 가 나옵니다.
(JSON + webm 변환까지 Colab 에서 끝내는 버전은 다음 단계.)

**순서**: 런타임을 GPU(T4 이상)로 바꾸고, 셀을 위에서부터 차례로 실행합니다.

**CLI 로 돌릴 때** (`google-colab-cli`): 폼 값 대신 환경변수 `DYAD_*` 가 우선합니다. 자세한 건 `colab/README.md`.

> AI 로 만든 채보임을 항상 밝히세요. 모델 제작자의 규칙입니다.


In [ ]:
#@title 1. 환경 준비 { display-mode: "form" }
#@markdown Mapperatorinator 를 받고 의존성을 설치합니다. 세션당 한 번만 실행하면 됩니다.
#@markdown 모델 제작자의 규칙: **AI 로 만든 채보임을 항상 밝힌다.** 동의하면 체크하고 실행하세요.
i_accept_the_rules = False # @param {type:"boolean"}
import os
if os.environ.get("DYAD_ACCEPT_RULES") == "1":
    i_accept_the_rules = True
assert i_accept_the_rules, "규칙에 동의한 뒤 실행하세요 (i_accept_the_rules)."

import subprocess, sys, shutil
from pathlib import Path

WORK = Path("/content")
REPO = WORK / "Mapperatorinator"
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/OliBomby/Mapperatorinator.git", str(REPO)], check=True)
os.chdir(REPO)

# 업스트림 노트북과 같은 핀. (torch 는 Colab 기본 것을 씁니다.)
def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
pip("transformers==4.57.3")
pip("hydra-core", "nnaudio")
pip("slider", "git+https://github.com/OliBomby/slider.git")
pip("rosu-pp-py==3.1.0")
pip("peft==0.18.1")

assert shutil.which("ffmpeg"), "ffmpeg 가 없습니다."
try:
    import torch
    assert torch.cuda.is_available(), "GPU 런타임이 아닙니다. 런타임 > 런타임 유형 변경 > GPU"
    print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    raise SystemExit("torch 가 없습니다.")

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from hydra import compose, initialize_config_dir
from osuT5.osuT5.event import ContextType
from inference import main as mapperatorinator_main

IN_DIR = WORK / "dyad-in"
OUT_DIR = WORK / "dyad-out"
IN_DIR.mkdir(exist_ok=True)
print("준비 완료")


In [ ]:
#@title 2. 음원과 정보 { display-mode: "form" }
#@markdown 음원 경로를 비워 두면 업로드 창이 뜹니다. mp3 / ogg / wav / flac / m4a 모두 됩니다.
audio_path = "" # @param {type:"string"}
#@markdown 자켓은 선택입니다. 보통은 비워 두고 로컬 `songs-src/<id>/` 에 `jacket.png|jpg|webp` 를 직접 넣는 편이 빠릅니다.
jacket_path = "" # @param {type:"string"}
#@markdown 곡 폴더 이름(영문 소문자·숫자·하이픈). 비워 두면 제목에서 만듭니다.
song_id = "" # @param {type:"string"}
#@markdown 제목·아티스트. 비워 두면 파일 이름 `아티스트 - 제목.mp3` 에서 추측합니다.
title = "" # @param {type:"string"}
artist = "" # @param {type:"string"}

import os, re, shutil, unicodedata
from pathlib import Path

def env(name, value):
    v = os.environ.get(name)
    return v if v not in (None, "") else value

audio_path = env("DYAD_AUDIO", audio_path)
jacket_path = env("DYAD_JACKET", jacket_path)
song_id = env("DYAD_SONG_ID", song_id)
title = env("DYAD_TITLE", title)
artist = env("DYAD_ARTIST", artist)

AUDIO_EXTS = {".mp3", ".ogg", ".wav", ".flac", ".m4a", ".aac", ".opus", ".webm"}

def upload_one(kinds, label):
    from google.colab import files
    picked = list(files.upload().keys())
    if not picked:
        raise SystemExit(f"{label} 파일이 없습니다.")
    if len(picked) > 1:
        print("여러 개가 올라와서 첫 번째만 씁니다.")
    name = picked[0]
    if Path(name).suffix.lower() not in kinds:
        raise SystemExit(f"{label}: 지원하지 않는 형식 {name}")
    dest = IN_DIR / name
    shutil.move(name, dest)
    return str(dest)

if not audio_path:
    audio_path = upload_one(AUDIO_EXTS, "음원")
audio_path = str(Path(audio_path).expanduser().resolve())
assert Path(audio_path).is_file(), f"음원을 찾을 수 없습니다: {audio_path}"
assert Path(audio_path).suffix.lower() in AUDIO_EXTS, f"지원하지 않는 형식: {audio_path}"

if jacket_path:
    jacket_path = str(Path(jacket_path).expanduser().resolve())
    assert Path(jacket_path).is_file(), f"자켓을 찾을 수 없습니다: {jacket_path}"

stem = Path(audio_path).stem
if not title or not artist:
    m = re.match(r"^\s*(.+?)\s+-\s+(.+?)\s*$", stem)
    guess_artist, guess_title = (m.group(1), m.group(2)) if m else ("Unknown Artist", stem)
    title = title or guess_title
    artist = artist or guess_artist

def slugify(s):
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    s = re.sub(r"[^a-z0-9]+", "-", s.lower()).strip("-")
    return s or "song"

song_id = slugify(song_id or title)
assert re.fullmatch(r"[a-z0-9][a-z0-9-]*", song_id), f"song_id 형식이 틀립니다: {song_id}"

print(f"음원   : {audio_path}")
print(f"자켓   : {jacket_path or '(없음, 로컬에서 넣기)'}")
print(f"id     : {song_id}")
print(f"제목   : {title}")
print(f"아티스트: {artist}")


In [ ]:
#@title 3. 채보 생성 (EASY / NORMAL / HARD) { display-mode: "form" }
#@markdown 난이도는 osu!taiko 스타 레이팅입니다. 대략 Kantan 1–2, Futsuu 2–3, Muzukashii 3–4, Oni 4–5.5.
easy_stars = 2.0 # @param {type:"number"}
normal_stars = 3.2 # @param {type:"number"}
hard_stars = 4.6 # @param {type:"number"}
#@markdown 모델. V32 가 최신. V32-mini 는 빠르지만 품질이 낮습니다.
model = "Mapperatorinator V32" # @param ["Mapperatorinator V32", "Mapperatorinator V32-mini", "Mapperatorinator V31"]
#@markdown 스타일 기준 연도 (2007–2024). 최신일수록 현대적인 패턴.
year = 2023 # @param {type:"integer"}
#@markdown 시드. -1 이면 무작위. 같은 시드 + 같은 설정이면 같은 결과.
seed = -1 # @param {type:"integer"}
#@markdown 샘플링 온도(낮을수록 보수적)와 CFG 강도(디스크립터를 얼마나 따를지).
temperature = 0.9 # @param {type:"slider", min:0.5, max:1.2, step:0.05}
cfg_scale = 1.0 # @param {type:"slider", min:1, max:5, step:0.1}
#@markdown taiko 스타일 디스크립터(선택). 예: `style/finisher-heavy`, `style/mono-heavy`, `expression/simple`, `style/clean`. 쉼표로 구분.
descriptors = "" # @param {type:"string"}
#@markdown 느리지만 정확한 타이밍 추정기(가변 BPM 곡에 유용). 티어마다 적용됩니다.
super_timing = False # @param {type:"boolean"}

import os, random, re, shutil, time
from pathlib import Path

def envf(name, value, cast):
    v = os.environ.get(name)
    return cast(v) if v not in (None, "") else value

easy_stars = envf("DYAD_EASY", easy_stars, float)
normal_stars = envf("DYAD_NORMAL", normal_stars, float)
hard_stars = envf("DYAD_HARD", hard_stars, float)
model = envf("DYAD_MODEL", model, str)
year = envf("DYAD_YEAR", year, int)
seed = envf("DYAD_SEED", seed, int)
temperature = envf("DYAD_TEMPERATURE", temperature, float)
cfg_scale = envf("DYAD_CFG", cfg_scale, float)
descriptors = envf("DYAD_DESCRIPTORS", descriptors, str)
super_timing = envf("DYAD_SUPER_TIMING", super_timing, lambda v: v.lower() in ("1", "true", "yes"))

config_name = model.split(" ")[-1].lower()          # v32 / v32-mini / v31
descriptor_list = [d.strip() for d in descriptors.split(",") if d.strip()]
if seed < 0:
    seed = random.randint(0, 2**16)
print(f"seed = {seed}")

# 모델은 pydub(ffmpeg) 로 읽으므로 원본 형식 그대로 넘깁니다. 손실 재인코딩 없음.
model_audio = Path(audio_path)

SONG_OUT = OUT_DIR / song_id
if SONG_OUT.exists():
    shutil.rmtree(SONG_OUT)
SONG_OUT.mkdir(parents=True)

# 티어 → (버전 이름, 스타, OD, HP). OD/HP 는 DYAD 판정에 안 쓰이지만 taiko 관례대로 둡니다.
TIERS = [
    ("hard", "Hard", hard_stars, 6, 6),
    ("normal", "Normal", normal_stars, 5, 5),
    ("easy", "Easy", easy_stars, 4, 4),
]

def make_conf(tier, version, stars, od, hp):
    with initialize_config_dir(version_base="1.1", config_dir=str(REPO / "configs" / "inference")):
        conf = compose(config_name=config_name)
    conf.audio_path = str(model_audio)
    conf.output_path = str(SONG_OUT / f"raw-{tier}")
    conf.beatmap_path = str(reference) if reference else ""
    conf.gamemode = 1                               # osu!taiko
    conf.difficulty = float(stars)
    conf.year = int(year)
    conf.hitsounded = True
    conf.hp_drain_rate = hp
    conf.overall_difficulty = od
    conf.circle_size = 5
    conf.approach_rate = 5
    conf.slider_multiplier = 1.4
    conf.slider_tick_rate = 1
    conf.descriptors = descriptor_list
    conf.negative_descriptors = []
    conf.cfg_scale = float(cfg_scale)
    conf.temperature = float(temperature)
    conf.seed = int(seed)
    conf.title = title
    conf.artist = artist
    conf.creator = "Mapperatorinator"
    conf.version = version
    conf.export_osz = False
    conf.add_to_beatmap = False
    # T4 has no bf16: fp16 keeps the fast decode loop instead of falling back to fp32.
    import torch
    conf.precision = "bf16" if torch.cuda.is_bf16_supported() else "fp16"
    # V32 의 taiko 전용 체크포인트는 템플릿이 하나뿐입니다: in=[] → out=[TIMING, MAP, SV].
    # 그래서 티어마다 타이밍을 새로 추정합니다 (같은 시드 + 같은 음원이라 실질적으로 같게 나옵니다).
    conf.in_context = []
    conf.output_type = [ContextType.TIMING, ContextType.MAP, ContextType.SV]
    conf.super_timing = bool(super_timing)
    return conf

def fix_osu(path, version):
    """DYAD 폴더 규칙에 맞게 헤더만 손봅니다. 히트오브젝트·타이밍은 그대로."""
    text = path.read_text(encoding="utf-8")
    assert re.search(r"^Mode:\s*1\s*$", text, re.M), f"{path.name}: Mode 가 1(taiko)이 아닙니다"
    text = re.sub(r"^AudioFilename:.*$", "AudioFilename: song.wav", text, flags=re.M)
    text = re.sub(r"^Version:.*$", f"Version:{version}", text, flags=re.M)
    text = re.sub(r"^Title:.*$", f"Title:{title}", text, flags=re.M)
    text = re.sub(r"^TitleUnicode:.*$", f"TitleUnicode:{title}", text, flags=re.M)
    text = re.sub(r"^Artist:.*$", f"Artist:{artist}", text, flags=re.M)
    text = re.sub(r"^ArtistUnicode:.*$", f"ArtistUnicode:{artist}", text, flags=re.M)
    path.write_text(text, encoding="utf-8")

def stats(path):
    text = path.read_text(encoding="utf-8")
    body = text.split("[HitObjects]", 1)[1] if "[HitObjects]" in text else ""
    notes = rolls = spinners = 0
    for line in body.splitlines():
        parts = line.strip().split(",")
        if len(parts) < 5 or not parts[0].lstrip("-").isdigit():
            continue
        t = int(parts[3])
        if t & 2: rolls += 1
        elif t & 8: spinners += 1
        else: notes += 1
    return notes, rolls, spinners

results = {}
t0 = time.time()
for tier, version, stars, od, hp in TIERS:
    print(f"\n=== {version} ({stars}★) ===")
    conf = make_conf(tier, version, stars, od, hp)
    _, result_path = mapperatorinator_main(conf)
    dest = SONG_OUT / f"{tier}.osu"
    shutil.move(str(result_path), dest)
    fix_osu(dest, version)
    results[tier] = dest
    n, r, s = stats(dest)
    print(f"{version}: 노트 {n}, 드럼롤 {r}, 스피너 {s}  ({time.time() - t0:.0f}s 경과)")

for tier, *_ in TIERS:
    shutil.rmtree(SONG_OUT / f"raw-{tier}", ignore_errors=True)
print("\n생성 완료:", *[p.name for p in results.values()])


In [ ]:
#@title 4. DYAD 폴더로 묶기 { display-mode: "form" }
#@markdown `song.wav`(44.1kHz 16-bit) 와 `song.json` 을 채워 `dyad-<id>.zip` 으로 내려받습니다. 자켓은 로컬에서 넣습니다.
import json, os, shutil, subprocess, sys, zipfile, hashlib
from pathlib import Path

wav = SONG_OUT / "song.wav"
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", audio_path, "-vn",
                "-ar", "44100", "-ac", "2", "-c:a", "pcm_s16le", "-fflags", "+bitexact", str(wav)], check=True)

# 자켓은 주어졌을 때만, 원본 형식 그대로 넣습니다 (빌드가 png/jpg/webp/avif 를 받습니다).
if jacket_path:
    shutil.copyfile(jacket_path, SONG_OUT / f"jacket{Path(jacket_path).suffix.lower()}")

(SONG_OUT / "song.json").write_text(
    json.dumps({"title": title, "artist": artist, "audioOffset": 0}, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

bundle = OUT_DIR / f"dyad-{song_id}.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(SONG_OUT.iterdir()):
        z.write(p, f"{song_id}/{p.name}")
size_mb = bundle.stat().st_size / 1e6
print(f"묶음: {bundle}  ({size_mb:.1f} MB)")
print("풀어서 songs-src/ 에 넣고 `npm run songs:build`")

if os.environ.get("DYAD_NO_DOWNLOAD") != "1":
    try:
        from google.colab import files
        files.download(str(bundle))
    except Exception as e:
        print("브라우저 다운로드를 열지 못했습니다. CLI 라면: colab download", bundle, "./")
